# Ансамблі


In [1]:
import os, re, json, pickle, string
from typing import List
import numpy as np
import pandas as pd

files = [os.path.join(d, f) for d, _, fs in os.walk('/kaggle/input') for f in fs]
def find(name):
    found = [f for f in files if os.path.basename(f) == name]
    if not found:
        raise FileNotFoundError(f'{name} не знайдено в /kaggle/input. Доступні файли: {[os.path.basename(f) for f in files][:30]}')
    return found[0]

def load_preds(name):
    path = f'/kaggle/working/preds_{name}.pkl'
    if not os.path.exists(path):
        path = find(f'preds_{name}.pkl')
    with open(path, 'rb') as f:
        return pickle.load(f)

In [2]:
with open(find('prepared.pkl'), 'rb') as f:
    prepared = pickle.load(f)

val_text, lab_text, test_text = prepared['val_text'], prepared['lab_text'], prepared['test_text']

In [3]:
def hard_processing(text):
    text = re.sub(r"[{re.escape(string.punctuation)}]", "", text)
    text = re.sub(r'\d', '', text)
    text = re.sub(r'\b\w\b\s?', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text.lower()

def comp_metric(y_true: List[List[str]], y_pred: List[List[str]]):
    assert len(y_true) == len(y_pred)
    tp, fp, fn, p = 0.0, 0.0, 0.0, 0.0
    for t, pr in zip(y_true, y_pred):
        t = set([hard_processing(s) for s in t])
        pr = set([hard_processing(s) for s in pr])
        tp += len(t & pr); fp += len(pr - t); fn += len(t - pr); p += len(t)
    if tp + fp == 0:
        precision = 1.0 if p == 0 else 0.0
    else:
        precision = tp / (tp + fp)
    if tp + fn == 0:
        recall = 1.0 if p == 0 else 0.0
    else:
        recall = tp / (tp + fn)
    f1 = 0 if precision + recall == 0 else 2 * precision * recall / (precision + recall)
    return round(f1, 8)

EMOJI = re.compile('[\U0001F000-\U0001FFFF\u2600-\u27BF\uFE0F]')
POST_RULES = set()

def clean_span(s, rules):
    if 'newline' in rules:
        s = s.split('\n')[0]
    if 'emoji' in rules:
        s = EMOJI.sub('', s)
    s = s.strip()
    if 'strip' in rules:
        s = s.strip(' \t,.;:!?-–—"«»“”()')
    if 'letters' in rules and not re.search(r'[^\W\d_]', s):
        return None
    if 'upper' in rules and not any(w[0].isupper() for w in re.findall(r'\w+', s)):
        return None
    return s or None

def decode(text, offsets, probs, thr):
    is_loc = (1 - probs[:, 0]) > thr
    result, start, end = [], None, None
    for t, (t_start, t_end) in enumerate(offsets[:len(probs)]):
        new_loc = is_loc[t] and (probs[t, 1] >= probs[t, 2] or t == 0 or not is_loc[t - 1])
        if new_loc:
            if start is not None:
                result.append(text[start:end])
            start, end = t_start, t_end
        elif is_loc[t] and start is not None:
            end = t_end
        else:
            if start is not None:
                result.append(text[start:end])
            start, end = None, None
    if start is not None:
        result.append(text[start:end])
    if POST_RULES:
        result = [c for c in (clean_span(x, POST_RULES) for x in result) if c]
    return result

def score(probs_list, texts, offsets_list, golds, thr):
    preds = [decode(t, o, p, thr) for t, o, p in zip(texts, offsets_list, probs_list)]
    return comp_metric(golds, preds)

In [4]:
POST_RULES = {'newline', 'emoji', 'letters', 'upper'}

## Функції

In [5]:
results = []

def ensemble(names, new_name):
    saved = [load_preds(n) for n in names]
    kinds = {s['kind'] for s in saved}
    assert len(kinds) == 1
    kind = kinds.pop(); data = prepared[kind]
    probs = {split: [np.mean([s['probs'][split][i] for s in saved], axis=0) for i in range(len(saved[0]['probs'][split]))]
             for split in ['val', 'lab', 'test']}
    thrs = np.arange(0.1, 0.91, 0.05)
    val_scores = [score(probs['val'], val_text, data['val']['offsets'], data['val']['locs'], t) for t in thrs]
    thr = float(thrs[int(np.argmax(val_scores))])
    row = {'name': new_name, 'kind': kind, 'members': '+'.join(names), 'thr': round(thr, 2), 'val_f1': max(val_scores),
           'lab_f1': score(probs['lab'], lab_text, data['lab']['offsets'], data['lab']['locs'], thr), 'LB': None}
    results.append(row)
    with open(f'/kaggle/working/preds_{new_name}.pkl', 'wb') as f:
        pickle.dump({'probs': probs, 'thr': thr, 'kind': kind}, f)
    print(row)

In [6]:
def make_submission(name, path='/kaggle/working/submission.csv'):
    p = load_preds(name)
    offsets = prepared[p['kind']]['test']['offsets']
    preds = [decode(t, o, pr, p['thr']) for t, o, pr in zip(test_text, offsets, p['probs']['test'])]
    sub = pd.DataFrame({'text_id': prepared['word']['test'].get('text_id', range(len(test_text))),
                        'locations': [json.dumps(x, ensure_ascii=False) for x in preds]})
    sub.to_csv(path, index=False)
    print(sub.head(), '\nрядків:', len(sub), '| без локацій:', (sub['locations'] == '[]').mean().round(2))

## Ансамблі

In [7]:
ensemble(['word_lstm_full', 'word_gru_full'], 'ens_full')

{'name': 'ens_full', 'kind': 'word', 'members': 'word_lstm_full+word_gru_full', 'thr': 0.5, 'val_f1': 0.85026632, 'lab_f1': 0.57142857, 'LB': None}


In [8]:
ensemble(['word_lstm_full_lab', 'word_gru_full_lab'], 'ens_lab')

{'name': 'ens_lab', 'kind': 'word', 'members': 'word_lstm_full_lab+word_gru_full_lab', 'thr': 0.3, 'val_f1': 0.85245902, 'lab_f1': 1.0, 'LB': None}


In [9]:
ensemble(['word_lstm_full', 'word_gru_full', 'word_lstm_full_lab', 'word_gru_full_lab'], 'ens_four')

{'name': 'ens_four', 'kind': 'word', 'members': 'word_lstm_full+word_gru_full+word_lstm_full_lab+word_gru_full_lab', 'thr': 0.4, 'val_f1': 0.85423994, 'lab_f1': 0.79012346, 'LB': None}


## Результати

In [10]:
pd.DataFrame(results).sort_values('val_f1', ascending=False)

,name,kind,members,thr,val_f1,lab_f1,LB
2,ens_four,word,word_lstm_full+word_gru_full+word_lstm_full_la...,0.4,0.854240,0.790123,None
1,ens_lab,word,word_lstm_full_lab+word_gru_full_lab,0.3,0.852459,1.000000,None
0,ens_full,word,word_lstm_full+word_gru_full,0.5,0.850266,0.571429,None


## Сабміт

In [11]:
make_submission('ens_lab')

   text_id                                          locations
0        0                                                 []
1        1  ["Києві", "Шулявського шляхопроводу", "Шулявсь...
2        2  ["Гоголеве", "Миргородського району", "Полтавс...
3        3                                                 []
4        4                ["проспекті Академіка Глушкова, 9"] 
рядків: 477 | без локацій: 0.32
